# ETL-24 · Carga no PostgreSQL

- **Etapa:** ETL-24, Carga no PostgreSQL (Supabase)
- **Responsável:** Pedro Wellington
- **Issue:** ETL-24
- **Entrada:** `sql/ddl/etl_star_schema.sql`, `data/intermediarios/dims/*.parquet` e `data/intermediarios/fato.parquet`
- **Saída:** as 9 tabelas do modelo estrela no banco

Cria as tabelas pelo DDL e carrega as 8 dimensões e depois a fato. Pode rodar mais de uma vez: cada execução recria as tabelas e carrega do zero.

## 1. Setup

In [17]:
import os
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)   # roda a partir da raiz, para achar a pasta src/ e o arquivo .env

from src.conexao import obter_engine, testar_conexao

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"
DDL    = RAIZ / "sql" / "ddl" / "etl_star_schema.sql"
FATO   = INTERM / "fato.parquet"

# ordem de carga: as dimensões primeiro, a fato por último (ela aponta para as dimensões)
NOMES_DIMS = ['dim_tempo', 'dim_gravidade', 'dim_perfil_vitima', 'dim_transporte',
              'dim_localidade', 'dim_via', 'dim_profissao', 'dim_local_obito']

testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


True

## 2. Criar as tabelas pelo DDL

In [18]:
engine = obter_engine()

# lê o script e executa tudo de uma vez: ele apaga e recria as 9 tabelas
sql = DDL.read_text(encoding='utf-8')
conn = engine.raw_connection()
try:
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.commit()
finally:
    conn.close()

# confere o que existe no banco agora
pd.read_sql("SELECT table_name FROM information_schema.tables "
            "WHERE table_schema = 'public' ORDER BY table_name", engine)

,table_name
0,dim_gravidade
1,dim_local_obito
2,dim_localidade
3,dim_perfil_vitima
4,dim_profissao
5,dim_tempo
6,dim_transporte
7,dim_via
8,fato_pessoa_sinistro


## 3. Carregar as tabelas

In [19]:
import io

def ajustar_inteiros(df: pd.DataFrame) -> pd.DataFrame:
    """Colunas float que só têm números inteiros (ex.: 12.0) viram inteiro, senão o banco recusa."""
    df = df.copy()
    for c in df.select_dtypes(include='float').columns:
        if (df[c].dropna() % 1 == 0).all():
            df[c] = df[c].astype('Int64')
    return df

def carregar_tabela(nome: str, df: pd.DataFrame, lote: int = 100_000) -> None:
    """Envia o DataFrame para a tabela com COPY, em lotes de 100 mil linhas."""
    df = ajustar_inteiros(df)
    colunas = ', '.join(df.columns)
    conn = engine.raw_connection()
    try:
        with conn.cursor() as cur:
            for i in range(0, len(df), lote):
                buf = io.StringIO()
                df.iloc[i:i + lote].to_csv(buf, index=False, header=False)   # lote em CSV, na memória
                buf.seek(0)
                cur.copy_expert(f"COPY {nome} ({colunas}) FROM STDIN WITH (FORMAT csv)", buf)
        conn.commit()
    finally:
        conn.close()

In [20]:
# dimensões primeiro, a fato por último; cada tabela é conferida contra o parquet
for nome in NOMES_DIMS + ['fato_pessoa_sinistro']:
    arquivo = FATO if nome == 'fato_pessoa_sinistro' else DIMS / f'{nome}.parquet'
    df = pd.read_parquet(arquivo)
    carregar_tabela(nome, df)
    n_banco = pd.read_sql(f'SELECT COUNT(*) AS n FROM {nome}', engine)['n'][0]
    print(f'{nome:24} parquet: {len(df):>8} | banco: {n_banco:>8} | ok: {len(df) == n_banco}')

dim_tempo                parquet:     1121 | banco:     1121 | ok: True
dim_gravidade            parquet:        6 | banco:        6 | ok: True
dim_perfil_vitima        parquet:     1500 | banco:     1500 | ok: True
dim_transporte           parquet:       25 | banco:       25 | ok: True
dim_localidade           parquet:      647 | banco:      647 | ok: True
dim_via                  parquet:       10 | banco:       10 | ok: True
dim_profissao            parquet:      561 | banco:      561 | ok: True
dim_local_obito          parquet:        5 | banco:        5 | ok: True
fato_pessoa_sinistro     parquet:   806977 | banco:   806977 | ok: True


## 4. Resumo

**O que foi feito:** criação das 9 tabelas do modelo estrela no PostgreSQL (Supabase) pelo DDL da ETL-18 e carga das 8 dimensões e da fato.

**Como foi carregado:** `COPY` em lotes de 100 mil linhas, a partir dos parquets. As dimensões entram primeiro e a fato por último, porque ela aponta para as chaves delas.

**Rodar mais de uma vez:** o DDL começa apagando e recriando as tabelas, então cada execução carrega do zero e não duplica nada.

**Resultado:**

| Tabela | Linhas |
|---|---|
| dim_tempo | 1.121 |
| dim_gravidade | 6 |
| dim_perfil_vitima | 1.500 |
| dim_transporte | 25 |
| dim_localidade | 647 |
| dim_via | 10 |
| dim_profissao | 561 |
| dim_local_obito | 5 |
| fato_pessoa_sinistro | 806.977 |

**O que foi conferido:** a contagem de cada tabela no banco bate com a do parquet de origem.

**Observação:** colunas que vinham como float com valores inteiros (por exemplo 12.0) são convertidas antes da carga, senão o banco recusa. A senha do banco fica só no `.env`, que não vai para o Git.